# Ejercicio: Web Scraping
## Nombre: Joel Quilumba
### Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [1]:
from bs4 import BeautifulSoup

file = '/content/sample_data/rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [2]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [3]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [4]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [30]:
# Find all the links to other recipes
all_links = soup.find_all("a", href=True)

# Redes sociales y páginas de sistema a excluir
excluded_patterns = [
    'facebook.com', 'instagram.com', 'pinterest.com', 'tiktok.com',
    'youtube.com', 'flipboard.com', 'twitter.com',
    '/authentication/', '/account/', 'magazines.com', '/cook/'
]

recipe_urls = []
for link in all_links:
    href = link['href']

    # Filtro estricto:
    # 1. Debe contener '/recipe/' (patrón de AllRecipes para recetas individuales)
    # 2. No debe contener patrones de redes sociales o páginas de cuenta/revistas
    if "/recipe/" in href.lower():
        if not any(pattern in href.lower() for pattern in excluded_patterns):
            # Normalizar rutas relativas
            if href.startswith('/'):
                href = "https://www.allrecipes.com" + href

            if href not in recipe_urls:
                recipe_urls.append(href)

# Imprimir los enlaces filtrados
print(f"Se encontraron {len(recipe_urls)} enlaces de recetas genuinas:")
for url in recipe_urls:
    print(url)

Se encontraron 16 enlaces de recetas genuinas:
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/
https://www.allrecipes.

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [6]:
%%capture
!pip install -q langchain langchain-community sentence-transformers faiss-cpu google-generativeai

### 4.1 Preparar el Corpus y Embeddings
Convertiremos la información extraída (título, descripción, ingredientes e instrucciones) en documentos procesables.

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings

# Unificamos la información en un solo texto o lista de documentos
corpus_text = f"Receta: {title}\nDescripción: {description}\n\nIngredientes:\n" + "\n".join(ingredients) + "\n\nInstrucciones:\n" + "\n".join(instructions)

# Dividimos por secciones para una mejor recuperación
documents = [
    Document(page_content=f"Título y Descripción: {title}. {description}", metadata={"source": "info"}),
    Document(page_content=f"Ingredientes: {', '.join(ingredients)}", metadata={"source": "ingredients"}),
    Document(page_content=f"Instrucciones de preparación: {' '.join(instructions)}", metadata={"source": "instructions"})
]

# Inicializamos los embeddings (usando un modelo ligero de HuggingFace)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Creamos la base de datos vectorial (Vector Store)
vector_db = FAISS.from_documents(documents, embeddings)

print("Base de datos vectorial creada con éxito.")

/tmp/ipykernel_585/1750565235.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Base de datos vectorial creada con éxito.


In [33]:
import os
from bs4 import BeautifulSoup

def extract_recipe_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    s = BeautifulSoup(content, "html.parser")

    # Extraer título
    t_tag = s.find("meta", {"property": "og:title"})
    t = t_tag["content"] if t_tag else "Receta desconocida"

    # Extraer descripción
    d_tag = s.find("meta", {"name": "description"})
    d = d_tag["content"] if d_tag else ""

    # Extraer ingredientes
    ing_tags = s.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
    ing = [i.get_text().strip() for i in ing_tags]

    # Extraer instrucciones
    ins_tags = s.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
    ins = [i.get_text().strip() for i in ins_tags]

    return t, d, ing, ins

# Directorio donde están los archivos
dir_path = '/content/sample_data/'
new_docs = []

# Listamos archivos HTML (excluyendo el original si ya está)
files = [f for f in os.listdir(dir_path) if f.endswith('.html') and f != 'rotisserie-chicken.html']

print(f"Procesando {len(files)} nuevos archivos...")

for file_name in files:
    path = os.path.join(dir_path, file_name)
    try:
        t, d, ing, ins = extract_recipe_data(path)

        # Crear documentos para el vector store
        new_docs.append(Document(page_content=f"Receta: {t}. {d}", metadata={"source": file_name}))
        new_docs.append(Document(page_content=f"Ingredientes para {t}: {', '.join(ing)}", metadata={"source": file_name}))
        new_docs.append(Document(page_content=f"Instrucciones para {t}: {' '.join(ins)}", metadata={"source": file_name}))
        print(f"✔ Procesado: {t}")
    except Exception as e:
        print(f"✘ Error en {file_name}: {e}")

# Añadir a la base de datos existente
if new_docs:
    vector_db.add_documents(new_docs)
    print(f"\n¡Éxito! Se han añadido {len(new_docs)} fragmentos a la base de datos vectorial.")

Procesando 16 nuevos archivos...
✔ Procesado: Cilantro-Lime Grilled Chicken
✔ Procesado: Best Beer Can Chicken
✔ Procesado: Drunk Chicken
✔ Procesado: Darn Good Chicken
✔ Procesado: Good Frickin’ Paprika Chicken
✔ Procesado: Beer Can Chicken
✔ Procesado: Smoked Whole Chicken
✔ Procesado: Miso Honey Chicken
✔ Procesado: Grilled Spatchcocked Chicken
✔ Procesado: The Best Beer Can Chicken Ever
✔ Procesado: Beer Butt Chicken
✔ Procesado: Rosemary Buttermilk Chicken
✔ Procesado: Smoked Beer Butt Chicken
✔ Procesado: Buttermilk Barbecue Chicken
✔ Procesado: Easy Barbeque Chicken
✔ Procesado: Grilled Chicken Under a Brick

¡Éxito! Se han añadido 48 fragmentos a la base de datos vectorial.


In [ ]:
import pandas as pd

# Vamos a verificar cuántos documentos tenemos por fuente en la base de datos
# FAISS no permite contar directamente por metadatos de forma trivial, pero podemos
# hacer una búsqueda amplia para ver la variedad de fuentes.

test_query = "chicken recipe"
results = vector_db.similarity_search(test_query, k=20)

sources = set([doc.metadata.get('source') for doc in results])

print(f"Total de fuentes distintas encontradas en el top 20 de resultados: {len(sources)}")
print("Fuentes detectadas:")
for s in sources:
    print(f"- {s}")

print(f"\nEl corpus está listo con {len(vector_db.docstore._dict)} fragmentos de texto.")

### 4.2 Configurar el Modelo de Lenguaje (Gemini)
Configurar la `GOOGLE_API_KEY` en los secretos de Colab.

In [25]:
import google.generativeai as genai
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # Listar modelos disponibles para depuración si es necesario
    # for m in genai.list_models():
    #     if 'generateContent' in m.supported_generation_methods:
    #         print(m.name)

    # gemini-1.5-flash fue descontinuado por Google; usamos 2.5-flash
    model_name = 'gemini-2.5-flash'
    model = genai.GenerativeModel(model_name)
    print(f"Modelo {model_name} configurado correctamente.")
except Exception as e:
    print(f"Error: {e}. Asegúrate de tener configurada la GOOGLE_API_KEY.")

Modelo gemini-2.5-flash configurado correctamente.


### 4.3 Función de Consulta RAG
Esta función busca la información relevante en el corpus y genera una respuesta usando Gemini.

In [41]:
def ask_recipe(question):
    # Aumentamos k a 10 para asegurar que cubrimos instrucciones detalladas
    docs = vector_db.similarity_search(question, k=10)

    context_list = []
    for d in docs:
        source = d.metadata.get('source', 'desconocida')
        context_list.append(f"[Fuente: {source}]: {d.page_content}")

    context = "\n---\n".join(context_list)

    prompt = f"""
    Eres un experto chef. Responde basándote EXCLUSIVAMENTE en el contexto proporcionado.
    Busca minuciosamente tiempos, temperaturas e ingredientes dentro de los fragmentos.

    Contexto:
    {context}

    Pregunta: {question}
    """

    response = model.generate_content(prompt)
    return response.text

# Consulta re-intentada
pregunta = "¿Cuánto tiempo debe marinarse el Cilantro-Lime Grilled Chicken?"
print(f"Pregunta: {pregunta}")
try:
    print(f"Respuesta: {ask_recipe(pregunta)}")
except Exception as e:
    print(f"Error: {e}")

Pregunta: ¿Cuánto tiempo debe marinarse el Cilantro-Lime Grilled Chicken?
Respuesta: El Cilantro-Lime Grilled Chicken debe marinarse en el refrigerador de 30 minutos a toda la noche.


### 4.4 Verificación Final del Corpus
Este bloque confirma que la base de datos vectorial ha integrado correctamente los fragmentos de todas las recetas procesadas.

In [35]:
import pandas as pd

# Realizamos una búsqueda amplia para verificar la diversidad de fuentes en el corpus
test_query = "chicken recipe"
results = vector_db.similarity_search(test_query, k=30)

sources = set([doc.metadata.get('source') for doc in results])

print(f"--- Resumen del Corpus ---")
print(f"Total de fuentes únicas detectadas en la búsqueda: {len(sources)}")
print(f"Total de fragmentos en el almacén: {len(vector_db.docstore._dict)}")
print("\nFuentes incluidas en el corpus:")
for s in sorted(list(sources)):
    print(f"- {s}")

--- Resumen del Corpus ---
Total de fuentes únicas detectadas en la búsqueda: 13
Total de fragmentos en el almacén: 99

Fuentes incluidas en el corpus:
- 14531_beer-butt-chicken.html
- 19944_drunk-chicken.html
- 221093_good-frickin-paprika-chicken.html
- 228070_the-best-beer-can-chicken-ever.html
- 238575_cilantro-lime-grilled-chicken.html
- 258659_rosemary-buttermilk-chicken.html
- 274724_grilled-spatchcocked-chicken.html
- 275044_grilled-chicken-under-a-brick.html
- 275062_buttermilk-barbecue-chicken.html
- 281255_smoked-whole-chicken.html
- 34957_easy-barbeque-chicken.html
- 8998_darn-good-chicken.html
- ingredients
